# 案例：需求理解与事实推理

## 核心学习目标

本案例的核心知识点是基于提示词工程的思维链推理。通过在提示词中设置明确的分析任务和输出要求，引导模型按照预设推理思路理解用户需求，并生成分步骤的事实分析结果。

## 案例设计

为了突出这一机制，案例设计四类事实预调查方法，将相关信息划分为“已给出或已验证的事实”“需要查阅的事实”“需要推导的事实”和“有根据的猜测”。这种划分能够帮助模型识别已知条件、外部信息需求、待推导内容与不确定性信息。

**学习重点**：
1. 理解如何将需求理解与事实推理过程转化为具体的思维链提示词。
2. 使用 **LangChain `PromptTemplate`** 渲染完整提示词，并观察四类事实分析框架在模型输出中的体现。
3. 使用 **`ChatOpenAI`** 调用模型，验证模型能否按照预设框架生成结构化的分析结果。


## 0. 环境

本案例使用仓库根目录的共享 `.venv`。运行 notebook 前，请在仓库根目录执行 `uv pip install -r Chapter-2/requirements.txt`，并将 notebook 内核设置为该 `.venv` 对应的 Python 解释器。

In [ ]:
# 在 Windows 环境中将标准输出流设置为 UTF-8，避免输出中文或特殊字符时发生编码错误（可选）
import sys

if sys.platform == "win32":
    for _s in (sys.stdout, sys.stderr):
        try:
            _s.reconfigure(encoding="utf-8")
        except Exception:
            pass

In [3]:
from datetime import datetime
print("当前时间:", datetime.now().astimezone().strftime("%Y-%m-%d %H:%M:%S %z"))

当前时间: 2026-11-14 17:01:07 +0800


In [ ]:
# 用户请求
task = """请帮我规划 2027 年 5 月 1 日至 5 日的旅行，两个人，想去上海、苏州和杭州，还没确定城市游览顺序。希望包含西湖，酒店每晚不超过 800 元，尽量不要安排夜间跨城交通。请提供路线、酒店、天气和美食攻略。"""

## 1. 思维链提示词模板

该提示词将需求理解拆分为四类事实分析任务：
- 提取用户请求中已经明确给出的条件
- 识别需要进行外部查询的信息
- 确定需要通过计算或逻辑得出的结论
- 以及标记尚未验证的判断。
通过限定输出标题和内容范围，模型可以按照预设思维链生成结构化结果。

模板中占位符 `{task}`，用于传入用户的自然语言任务。

In [ ]:
FACTS_PROMPT = """在处理用户任务之前，请先完成以下信息分析。回答应准确、完整，并明确区分已知信息、待查证信息、推导结果与不确定性判断。

用户输入任务：

{task}

以下是预调查问题：

    1. 请列出请求中明确给出的任何具体事实或数据。可能没有任何此类信息。
    2. 请列出可能需要查阅的任何事实，以及具体可以在哪里找到这些信息。在某些情况下，请求本身会提及权威来源。
    3. 请列出可能需要推导的任何事实（例如，通过逻辑演绎、模拟或计算得出）。
    4. 请列出从记忆中回忆出的任何事实、直觉、经过充分推理的猜测等。

在回答此调查时，请记住，"事实"通常是具体的名称、日期、统计数据等。您的回答应使用以下标题：

    1. 已给出或已验证的事实
    2. 需要查阅的事实
    3. 需要推导的事实
    4. 有根据的猜测

不要在你的回复中包含其他任何标题或部分。在被要求之前，不要列出下一步行动或计划。
"""


## 2. 用 LangChain 封装 完整提示词

在本例中，`PromptTemplate.from_template` 与 Python 的 `FACTS_PROMPT.format(task=…)` 会生成相同的字符串。使用 `PromptTemplate` 便于后续集成 LCEL 并进行链式调试。

In [ ]:
from langchain_core.prompts import PromptTemplate

facts_template = PromptTemplate.from_template(FACTS_PROMPT)
full_prompt = facts_template.format(task=task.strip())


assert full_prompt == FACTS_PROMPT.format(task=task.strip())

## 3. 展示完整的思维链提示词

下面展示编排器在第一步发送给模型的**完整用户消息**。如仅需展示部分内容，可使用 `full_prompt[:2000]` 进行截取。

In [ ]:
from IPython.display import display, Markdown

display(Markdown(f"**字符数：** {len(full_prompt)}"))

In [ ]:
display(Markdown("```text\n" + full_prompt + "\n```"))

## 4.使用 LangChain `ChatOpenAI` 调用模型

如果希望实际调用 Qwen 执行上述思维链提示词，请先将仓库根目录的 `.env.example` 复制为 `.env`，并至少配置 `DASHSCOPE_API_KEY`。可通过 `DASHSCOPE_CHAT_MODEL` 指定模型，未配置时默认使用 `qwen3.7-plus`。以下代码会自动向上查找并加载 `.env`，无需在 notebook 中写入 API Key。

新版 Qwen 的 thinking 模式会显著增加响应时间。本案例只需要按既定框架整理信息，因此调用时显式关闭 thinking，并设置超时，避免长时间没有输出而被误认为内核卡死。

In [ ]:
import os

from dotenv import find_dotenv, load_dotenv
from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI

env_file = find_dotenv(usecwd=True)
if env_file:
    load_dotenv(env_file)

kw = dict(
    model=os.getenv("DASHSCOPE_CHAT_MODEL", "qwen3.7-plus"),
    temperature=0,
    base_url=os.getenv("DASHSCOPE_BASE_URL", "https://dashscope.aliyuncs.com/compatible-mode/v1"),
    api_key=os.environ["DASHSCOPE_API_KEY"],
    extra_body={"enable_thinking": False},
    timeout=60,
    max_retries=1,
)

llm = ChatOpenAI(**kw)
print(f"正在调用 {kw['model']}（已关闭 thinking），请稍候……")
resp = llm.invoke([HumanMessage(content=full_prompt)])

### 模型输出

In [ ]:
display(Markdown(resp.content if isinstance(resp.content, str) else str(resp.content)))

## 5. 小结

- 本例的**思维链提示词**由固定说明、通过 `{task}` 嵌入的任务，以及四个指定的输出标题组成，并作为单条用户消息提交。
- 在 **LangChain** 中，`PromptTemplate` 负责模板渲染，`ChatOpenAI` 负责向模型服务发起调用。